# 03 - Missing Values Handling

Each missing column has a specific reason for being missing, so each one is treated differently.

**Input:** `train_cleaned.csv` — 844,338 rows, 30 columns  
**Output:** `train_no_missing.csv` — same shape, 0 missing values

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [ ]:
df = pd.read_csv('../data/train_cleaned.csv', low_memory=False)
print(f'Shape: {df.shape}')
df.head()

## Missing Values Overview

In [ ]:
print(df.isnull().sum())

## 1. CompetitionDistance

Distance to the nearest competitor store in meters. Only 2,186 rows missing — a small fraction. The distribution is right-skewed with large outliers, so the **median** is a better fill than the mean.

In [ ]:
df["CompetitionDistance"] = df["CompetitionDistance"].fillna(df["CompetitionDistance"].median())

## 2. CompetitionOpenSinceMonth

The month (1–12) when the nearest competitor opened. Missing means the opening date is unknown. Since month is a cyclical categorical-like value (not a continuous measurement), the **mode** makes more sense than the mean.

In [ ]:
df["CompetitionOpenSinceMonth"] = df["CompetitionOpenSinceMonth"].fillna(df["CompetitionOpenSinceMonth"].mode()[0])

## 3. CompetitionOpenSinceYear

The year the competitor opened. Missing for the same stores as the month column. Filled with the **mode** (most common year in the dataset).

In [ ]:
df["CompetitionOpenSinceYear"] = df["CompetitionOpenSinceYear"].fillna(df["CompetitionOpenSinceYear"].mode()[0])

## 4. Promo2SinceWeek, Promo2SinceYear, PromoInterval, Promo2OpenWeeks

These are only populated for stores that run Promo2 (`Promo2 == 1`). For the other 423K rows the data isn't missing — Promo2 just doesn't apply to those stores. So numeric columns get **0** and `PromoInterval` gets **"None"**.

In [ ]:
no_promo2 = df["Promo2"] == 0

df.loc[no_promo2, "Promo2SinceWeek"] = 0
df.loc[no_promo2, "Promo2SinceYear"] = 0
df.loc[no_promo2, "Promo2OpenWeeks"] = 0
df.loc[no_promo2, "PromoInterval"]   = "None"

## 5. CompetitionOpenMonths

Number of months since the nearest competitor opened. Missing for the same rows as the competition date columns. Distribution is right-skewed, so filled with the **median**.

In [ ]:
df["CompetitionOpenMonths"] = df["CompetitionOpenMonths"].fillna(df["CompetitionOpenMonths"].median())

## Final Check

In [ ]:
remaining = df.isnull().sum().sum()
print(f'Total missing values: {remaining}')
assert remaining == 0, 'Still has missing values!'
print(f'Shape: {df.shape}')
print('All clean.')

## Save

In [ ]:
df.to_csv('../data/train_no_missing.csv', index=False)
print(f'Saved → train_no_missing.csv  |  shape: {df.shape}')